In [ ]:
from IPython.display import HTML
HTML('''
    <style> body {font-family: "Roboto Condensed Light", "Roboto Condensed";} h2 {padding: 10px 12px; background-color: #E64626; position: static; color: #ffffff; font-size: 40px;} .text_cell_render p { font-size: 15px; } .text_cell_render h1 { font-size: 30px; } h1 {padding: 10px 12px; background-color: #E64626; color: #ffffff; font-size: 40px;} .text_cell_render h3 { padding: 10px 12px; background-color: #0148A4; position: static; color: #ffffff; font-size: 20px;} h4:before{ 
    content: "@"; font-family:"Wingdings"; font-style:regular; margin-right: 4px;} .text_cell_render h4 {padding: 8px; font-family: "Roboto Condensed Light"; position: static; font-style: italic; background-color: #FFB800; color: #ffffff; font-size: 18px; text-align: center; border-radius: 5px;}input[type=submit] {background-color: #E64626; border: solid; border-color: #734036; color: white; padding: 8px 16px; text-decoration: none; margin: 4px 2px; cursor: pointer; border-radius: 20px;}</style>
''')

# Electricity Sector Data Integration & Augmentation

This notebook builds a **spatial data warehouse** for the Australian National Electricity Market (NEM) by integrating data from three government sources:

1. **NGER** (National Greenhouse and Energy Reporting) — 10 years of facility-level emissions and electricity production
2. **CER** (Clean Energy Regulator) — Accredited, committed, and probable power station registries
3. **ABS** (Australian Bureau of Statistics) — State-level economic indicators

The pipeline: **Acquire** → **Clean & Integrate** → **Augment with Geocoding** → **Store in DuckDB with Spatial Support**.

In [ ]:
import pandas as pd
import numpy as np
import requests
import re
import time as t
import matplotlib.pyplot as plt

## Data Acquisition

### Data Sources

We pull from three distinct sources, each requiring a different ingestion strategy:

| Source | Format | Method | Years |
|---|---|---|---|
| NGER | JSON API (yearly endpoints) | `requests` → `pd.DataFrame` | 2014–2023 |
| CER | CSV files (3 statuses) | `pd.read_csv` from URL | Current registry |
| ABS | Excel (.xlsx) Table 1 | `pd.read_excel` with skiprows | 2011–2024 |

### NGER

In [ ]:
urls = {
     # Year API endpoint (JSON). Each URL returns one year's table.
    2014: "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0075?select%3D%2A",
    2015: "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0076?select%3D%2A",
    2016: "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0077?select%3D%2A",
    2017: "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0078?select%3D%2A",
    2018: "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0079?select%3D%2A",
    2019: "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0080?select%3D%2A",
    2020: "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0081?select%3D%2A",
    2021: "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0082?select%3D%2A",
    2022: "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0083?select%3D%2A",
    2023: "https://api.cer.gov.au/datahub-public/v1/api/ODataDataset/NGER/dataset/ID0243?select%3D%2A"
}

# collect one DataFrame per year
frames = []

for year, url in urls.items():
    res = requests.get(url).json()
    
    # Convert JSON to a pandas DataFrame
    df = pd.DataFrame(res)
    
    #Add the year as an explicit column (so we can stack years later)
    df["year"] = year
    frames.append(df)

NGER = pd.concat(frames, ignore_index = True)
NGER.to_csv("NGER_raw.csv", index = False)
#print(NGER.head(5))

In [ ]:
print(NGER.shape)

### CER

In [ ]:
# CER: collect three public CSVs (accredited / committed / probable)

BASE = "https://cer.gov.au"
hrefs = {1:"/document/power-stations-and-projects-accredited",
         2:"/document/power-stations-and-projects-committed",
         3:"/document/power-stations-and-projects-probable"}

# collect one DataFrame per document
frames = []
for status, href in hrefs.items():
    # build absolute URL
    url = BASE + href
    
    df = pd.read_csv(url)
    # tag the source category
    df["status"] = status
    
    frames.append(df)

# stack all three lists into one table
CER = pd.concat(frames, ignore_index=True)

CER.to_csv("CER_combined_raw.csv", index = False)
#print(CER.head(5))

In [ ]:
print(CER.shape)

### ABS

In [ ]:
#ABS: download the official Excel and extract Table 1

url = "https://www.abs.gov.au/methodologies/data-region-methodology/2011-24/14100DO0003_2011-24.xlsx"

df = pd.read_excel(url, sheet_name="Table 1", skiprows=6)

df.to_csv("ABS_Economy&Industry_State_table1_raw.csv", index = False)

## Data Integration and Cleaning

### NGER

In [ ]:
nger = pd.read_csv("NGER_raw.csv")

#Normalize column names to detect same fields across years
def _norm(name: str) -> str:
    return name.lower().replace('-', '').replace(' ', '')

orig_cols = list(nger.columns)   # original headers
norm_cols = [_norm(c) for c in orig_cols]    # normalized headers


out = pd.DataFrame(index=nger.index)

#treat column as numeric if >= 70% of non-null values can coerce to numbers
def looks_numeric(series: pd.Series) -> bool:
    s = pd.to_numeric(series, errors='coerce')
    nonnull = series.notna().sum()
    return s.notna().sum() >= 0.7 * nonnull if nonnull > 0 else False

#Collapse columns that normalize to the same name
for raw, norm in zip(orig_cols, norm_cols):
    col = nger[raw]
    if norm not in out.columns:
        out[norm] = col
        # duplicate for the same field name combine with existing
    else:
        left = out[norm]
        right = col
        left_is_num = looks_numeric(left)
        right_is_num = looks_numeric(right)
        # both numeric ： coerce to numbers
        if left_is_num and right_is_num:
            out[norm] = pd.to_numeric(left, errors='coerce').fillna(0) + \
                        pd.to_numeric(right, errors='coerce').fillna(0)
        else:
            # text：prefer non-null/right when left is NA; otherwise keep left
            out[norm] = left.combine_first(right)
# Utilities to merge a specific numeric/text pair, then drop the duplicate
def add_then_drop(df: pd.DataFrame, left: str, right: str):
    if left in df.columns and right in df.columns:
        df[left] = pd.to_numeric(df[left], errors='coerce').fillna(0) + \
                   pd.to_numeric(df[right], errors='coerce').fillna(0)
        df.drop(columns=[right], inplace=True)

# Hand-tuned numeric documents
add_then_drop(out, 'scope1tco2e', 'totalscope1emissionstco2e')
add_then_drop(out, 'scope2tco2e', 'totalscope2emissionstco2e')
add_then_drop(out, 'emissionintensitytmwh', 'emissionintensitytco2emwh')
add_then_drop(out, 'totalemissionstco2e', 'totalscope2emissionstco2e2')

def merge_text_then_drop(df: pd.DataFrame, left: str, right: str):
    if left in df.columns and right in df.columns:
        l = df[left].astype("string")
        r = df[right].astype("string")

        def _merge(a, b):
            if pd.isna(a) or a == "<NA>":
                # left empty then take right
                return b if (not pd.isna(b) and b != "<NA>") else pd.NA
            if pd.isna(b) or b == "<NA>":
                return a
            a_str = str(a)
            b_str = str(b)
            # if right text is already contained in left, keep left; else union
            return a_str if b_str.strip() in a_str else f"{a_str} | {b_str}"

        df[left] = [ _merge(a, b) for a, b in zip(l, r) ]
        df.drop(columns=[right], inplace=True)
# Hand-tuned text documents
merge_text_then_drop(out, 'reportingentity', 'controllingcorporation')
merge_text_then_drop(out, 'gridconnected', 'gridconnected2')

nger = out

In [ ]:
# Minimum fields required for analysis and later joins
base_cols = ["state", "year", "totalemissionstco2e", "electricityproductionmwh"]
# Keep only rows where these key columns are present
nger_clean = nger.dropna(subset=base_cols, how="any")

# If a'type'exists, remove corporate totals ('C') to avoid double-counting
## ('C' = company-level sum across all facilities; we want facility-level records)
if "type" in nger_clean.columns:
    nger_clean = nger_clean[nger_clean["type"] != "C"]

nger_clean.to_csv("NGER_done.csv",index=False)

**Cleaning summary for NGER:**
- Normalized column names by lowercasing and removing whitespace/dashes
- Merged duplicate columns (e.g., two different spellings of *Scope 1 emissions*) by detecting numeric vs text content
- Removed corporate-level totals (`type == "C"`) to avoid double-counting facilities
- Kept only rows with non-null values in state, year, emissions, and electricity production

### CER

In [ ]:
cer = pd.read_csv("CER_combined_raw.csv")

# Standardize headers so near-duplicates can be detected
def _norm(name: str) -> str:
    return name.lower().strip()
   
orig_cols = list(cer.columns)    # original headers
norm_cols = [_norm(c) for c in orig_cols]    # normalized headers


out = pd.DataFrame(index=cer.index)

def looks_numeric(series: pd.Series) -> bool:
    s = pd.to_numeric(series, errors='coerce')
    nonnull = series.notna().sum()
    return s.notna().sum() >= 0.7 * nonnull if nonnull > 0 else False

# Collapse columns that normalize to the same field name
for raw, norm in zip(orig_cols, norm_cols):
    col = cer[raw]
    if norm not in out.columns:
        # first apparence take as the base column
        out[norm] = col
    else:
        # duplicate field ： merge with the existing column
        left = out[norm]
        right = col
        left_is_num = looks_numeric(left)
        right_is_num = looks_numeric(right)

        if left_is_num and right_is_num:
            # numeric
            out[norm] = pd.to_numeric(left, errors='coerce').fillna(0) + \
                        pd.to_numeric(right, errors='coerce').fillna(0)
        else:
            # text
            out[norm] = left.combine_first(right)

def add_then_drop(df: pd.DataFrame, left: str, right: str, numeric=False):
    if left in df.columns and right in df.columns:
        if numeric:
            df[left] = pd.to_numeric(df[left], errors='coerce').fillna(0) + \
                       pd.to_numeric(df[right], errors='coerce').fillna(0)
        else:
            df[left] = df[left].combine_first(df[right])
        df.drop(columns=[right], inplace=True)
        
# Hand-tuned documents
add_then_drop(out, 'power station name', 'project name', numeric=False)
add_then_drop(out, 'installed capacity (mw)', 'mw capacity', numeric=True)
add_then_drop(out, 'fuel source (s)', 'fuel source', numeric=False)

cer = out

In [ ]:
# Create a cleaned name for CER power stations
def clean_name(s: str) -> str:
    if pd.isna(s):
        return s
    x = str(s)
    # Cut off any suffix after a dash
    x = re.split(r"\s*[-–—]\s*", x, maxsplit=1)[0]
    # Remove anything inside parentheses
    x = re.sub(r"\s*\([^)]*\)\s*", " ", x)
    # Collapse repeated whitespace and trim
    x = re.sub(r"\s+", " ", x).strip()
    return x

col = "power station name"
if col in cer.columns:
    # Apply cleaner and store as a new canonical name field
    cer["power_station_name_clean"] = cer["power station name"].map(clean_name)
else:
    exit
#print(cer.columns.tolist())

In [ ]:
name_col = "power_station_name_clean"

# Normalize the cleaned name
cer[name_col] = (
    cer[name_col]
    .astype(str)
    .str.replace(r"stage", "", flags=re.IGNORECASE, regex=True) 
    .str.replace(r"\d+", "", regex=True)                     
    .str.strip()
)

# Build a boolean mask for rows that look like non-power entries and drop
mask = (
    cer[name_col].str.contains("company", case=False, na=False) |
    cer[name_col].str.contains("shopping centre", case=False, na=False)
)
# Keep only rows that are NOT matched by the mask
cer = cer[~mask]

In [ ]:
# Map state abbreviations to full names

STATE_FULLNAME = {
    'NSW': 'New South Wales',
    'VIC': 'Victoria', 
    'QLD': 'Queensland',
    'SA': 'South Australia',
    'WA': 'Western Australia',
    'TAS': 'Tasmania',
    'NT': 'Northern Territory',
    'ACT': 'Australian Capital Territory'
}
cer["state_full"] = cer["state"].map(STATE_FULLNAME).fillna(cer["state"])

# Regex patterns:
legal_suffix_re  = re.compile(
    r'\b(pty|proprietary|ltd|limited|inc|incorporated|'
    r'trading|group|holdings?|company|corporation|trust|unit\s*trust|fund|nominees?)\b\.?',
    re.I
)
# Trailing energy/technology descriptors to drop from names
energy_suffix_re = re.compile(
    r'\s*[-–—]?\s*(solar|wind|hydro|hydroelectric|battery|bess|pv|diesel|gas|biomass|'
    r'geothermal|hydrogen|power\s*station|sgu)\b.*',
    re.I
)
paren_re         = re.compile(r'\([^)]*\)')# (...) blocks
bracket_re       = re.compile(r'\[[^\]]*\]')# [...] blocks

def clean_name(s: str) -> str:
    if not str(s).strip():
        return ""
    x = str(s).strip()
    # remove bracketed notes
    x = paren_re.sub(" ", x)
    x = bracket_re.sub(" ", x)
    # remove legal/company suffixes
    x = energy_suffix_re.sub("", x)
    x = legal_suffix_re.sub("", x)
    # normalize punctuation/whitespace
    x = re.sub(r"\s*[-–—]\s*", ", ", x)
    x = re.sub(r"[|•·~`^_+=<>]", " ", x)
    x = re.sub(r"\s*,\s*,\s*", ", ", x)
    x = re.sub(r"\s+", " ", x).strip(" ,")
    return x
col = "power station name"
cer["power_station_name_clean"] = cer[col].map(clean_name)

# Drop obvious non-power entries
mask_keep = ~cer[col].str.contains(r"woolworths", case=False, na=False)
cer = cer.loc[mask_keep].copy()

# Build a tidy postcode and geocoding query string
postcode = cer["postcode"].astype(str).str.strip()
postcode = postcode.str.replace(r"\.0+$", "", regex=True)

cer["query_address"] = (
    cer["power_station_name_clean"].str.strip() + ", " +
    cer["state_full"].str.strip() + ", " +
    postcode + ", Australia"
).str.replace(", ,", ",", regex=False).str.strip(" ,")

# Save final cleaned CER for geocoding
cer.to_csv("CER_done.csv", index=False)

**Cleaning summary for CER:**
- Standardized column names across the three CSV files (accredited/committed/probable)
- Merged duplicate *power station name* and *installed capacity* columns
- Stripped legal suffixes (Pty Ltd, Trust, etc.) and energy descriptors from facility names
- Filtered out non-power entries (shopping centres, corporate offices)
- Built a geocoding query string: `{name}, {state}, {postcode}, Australia`

### ABS

In [ ]:
abs = pd.read_csv("ABS_Economy&Industry_State_table1_raw.csv")

# Columns we plan to use
keep_cols = [
    "Code",
    "Label",
    "Year",
    "Total number of businesses",
    "Total number of business entries",
    "Total number of business exits",
    "Electricity, gas, water and waste services (no.)",
    "Value of non-residential building ($m)",
    "Technicians and trades workers (no.)",
    "Electricity, gas, water and waste services (%)"
]

keep_cols = [c for c in keep_cols if c in abs.columns]
abs = abs[keep_cols].copy()

# Filter to state-level rows
abs["Code"] = pd.to_numeric(abs["Code"], errors="coerce")
abs = abs[abs["Code"].between(1, 9, inclusive="both")]
abs["Code"] = abs["Code"].astype("Int64").astype(str)

abs["Year"] = pd.to_numeric(abs["Year"], errors="coerce")
abs = abs[abs["Year"].between(2014, 2024, inclusive="both")]

abs.to_csv("ABS_done.csv", index=False)

## Data Augmentation

In [ ]:
# Geocoding CER records via OpenStreetMap Nominatim
BASE_URL = "https://nominatim.openstreetmap.org/search"
HEADERS  = {"User-Agent": "EnergyDataPipeline/1.0"}

def geocode_q(q):
    t.sleep(1.1)
    r = requests.get(BASE_URL, params={
        'q': q, 'countrycodes': 'au', 'format': 'jsonv2', 'limit': 1
    }, headers=HEADERS, timeout=30)
    r.raise_for_status()
    js = r.json()
     # take the first match
    if js:
        return float(js[0]['lat']), float(js[0]['lon'])
    return None, None

df = pd.read_csv("CER_done.csv")
df = df.head(345)
df["lat"], df["lon"] = None, None
ok = 0

# Look up each address string and write back coordinates
for i, q in enumerate(df["query_address"]):
    lat, lon = geocode_q(q)
    df.at[i, "lat"], df.at[i, "lon"] = lat, lon
    if lat is not None: ok += 1
print(f"Total: {len(df)}, matched: {ok}, ratio={ok/len(df):.2%}")

df.to_csv("cer_coordinates.txt", sep="\t", index=False)
df.to_csv("cer_coordinates.csv", index=False)

### Visualization

#### 1. NEGR： How emissions change over time

In [ ]:
nger = pd.read_csv("NGER_done.csv")

# Aggregate yearly totalemissionstco2e
df_trend = nger.groupby("year", as_index=False)["totalemissionstco2e"].sum()
plt.figure(figsize=(10,6))

#Bars = yearly total emission
bars = plt.bar(df_trend["year"], df_trend["totalemissionstco2e"]/1e6, color="skyblue", label="Total Emissions")

plt.plot(df_trend["year"], df_trend["totalemissionstco2e"]/1e6, color="darkblue", marker="o", linewidth=2, label="Trend")


#Titles, labels, legend, and light grid
plt.title("Total Emissions Trend (2014–2024)", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Total Emissions (million tCO₂-e)")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()


plt.savefig("NGER_total_emissions_trend.png", dpi=300)
plt.show()

#### 2. CER： Where power stations are located

### Geocoding with OpenStreetMap Nominatim

We use the free Nominatim API to convert facility addresses into lat/lon coordinates. The service has a **1 request/sec fair-use limit** (enforced by `time.sleep(1.1)` between calls).

The match rate is ~34% because many CER entries have incomplete addresses or are rooftop solar installations without a distinct street address. Records without coordinates are dropped before spatial database insertion.

In [ ]:

geo_path = "cer_coordinates.csv"
# Load geocoded points and read lat/lon as numbers and drop rows without them
df = pd.read_csv(geo_path, dtype=str, keep_default_na=False)
df["lat"] = pd.to_numeric(df.get("lat"), errors="coerce")
df["lon"] = pd.to_numeric(df.get("lon"), errors="coerce")
geo = df.dropna(subset=["lat", "lon"]).copy()

#Find the fuel column
fuel_col = next((c for c in df.columns if "fuel" in c.lower()), None)

def fuel_to_cat(s: str) -> str:
    t = (s or "").lower()
    if "solar" in t or "pv" in t: return "Solar"
    if "wind" in t:               return "Wind"
    if "hydro" in t:              return "Hydro"
    return "Other"

geo["category"] = geo[fuel_col].map(fuel_to_cat) if fuel_col else "Unknown"

# Simple scatter by category
fig, ax = plt.subplots(figsize=(9, 8))

for cat, sub in geo.groupby("category"):
    ax.scatter(sub["lon"], sub["lat"], s=12, label=cat)

ax.set_title("Power Stations in Australia (by category)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(loc="upper left", frameon=True) 
plt.tight_layout()
plt.show()

## Data Transformation and Storage

In [ ]:
#pip install jupysql duckdb-engine

In [ ]:
#%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [ ]:
# Load CER data with coordinates from a CSV processed by API
df_c = pd.read_csv("cer_coordinates.txt", sep="\t")
print(df_c.head())

In [ ]:
import duckdb

%load_ext sql
conn = duckdb.connect("database.db")
%sql conn --alias duckdb

In [ ]:
%%sql duckdb
INSTALL spatial;
LOAD spatial;

In [ ]:
#%pip install geopandas
#%pip install geoalchemy2

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon
from geoalchemy2 import Geometry, WKTElement

### Relational Schema Design

We normalize the merged data into a **star-like schema** with 7 tables:
![alt text](image.png)

- **Fact_Emission** — yearly emissions & production per facility, with FK to States, Facility, Grid, Primary_fuel, Economic_status
- **Fact_Projects** — CER power station registry with spatial geometry (WGS84)
- **Economic_status** — state × year economic indicators from ABS
- DuckDB **Spatial extension** enables geospatial queries on the `geom` column

In [ ]:
# convert coordinates to spatial data
df_c = df_c[df_c['lat'].notnull() & df_c['lon'].notnull()].copy()
df_c['geom'] = gpd.points_from_xy(df_c.lat, df_c.lon)
df_c = df_c.drop(columns=['lat', 'lon']) 
df_c.head(5)

In [ ]:
%sql SET python_scan_all_frames=true

In [ ]:
# convert to wkt
srid = 4326
df_c['geom'] = df_c['geom'].apply(lambda x: WKTElement(x.wkt, srid=srid))
df_c.head(5)

In [ ]:
%%sql
-- load cleaned data
DROP TABLE IF EXISTS nger;
CREATE TABLE nger AS SELECT * FROM nger_clean;
DROP TABLE IF EXISTS cer;
CREATE TABLE cer AS SELECT * FROM df_c;
DROP TABLE IF EXISTS abs;
CREATE TABLE abs AS SELECT * FROM abs;

In [ ]:
%%sql
-- further clean abs due to the need to insert into the schema
UPDATE abs
SET "Total number of businesses" = NULLIF("Total number of businesses", '-'),
    "Total number of business entries" = NULLIF("Total number of business entries", '-'),
    "Total number of business exits" = NULLIF("Total number of business exits", '-'),
    "Electricity, gas, water and waste services (no.)" = NULLIF("Electricity, gas, water and waste services (no.)", '-'),
    "Value of non-residential building ($m)" = NULLIF("Value of non-residential building ($m)", '-'),
    "Technicians and trades workers (no.)" = NULLIF("Technicians and trades workers (no.)", '-'),
    "Electricity, gas, water and waste services (%)" = NULLIF("Electricity, gas, water and waste services (%)", '-');

In [ ]:
%%sql 
-- further clean nger due to the need to insert into the schema
-- enable foreign key referencing Fact_projects
DELETE FROM nger WHERE year NOT BETWEEN 2016 AND 2024;

In [ ]:
%%sql
--create the schema
DROP TABLE IF EXISTS Fact_Emission;
DROP TABLE IF EXISTS Fact_Projects;
DROP TABLE IF EXISTS Economic_status;
DROP TABLE IF EXISTS States;
DROP TABLE IF EXISTS Facility;
DROP TABLE IF EXISTS Primary_fuel;
DROP TABLE IF EXISTS Grid;
DROP TABLE IF EXISTS States;

CREATE TABLE States (
    State_id INT primary key,
    State_abbr VARCHAR(3),
    State_full VARCHAR(50)
);

DROP TABLE IF EXISTS Facility;
CREATE TABLE Facility (
    Facility_id INT primary key,
    Facility_name VARCHAR(100) NOT NULL,
    Reporting_entity VARCHAR(100)
);

DROP TABLE IF EXISTS Primary_fuel;
CREATE TABLE Primary_fuel (
    Primary_fuel_id INT primary key,
    Primary_fuel_name VARCHAR(50) NOT NULL
);

DROP TABLE IF EXISTS Grid;
CREATE TABLE Grid (
    Grid_id INT primary key,
    Grid_name VARCHAR(50) NOT NULL
);

CREATE TABLE Economic_status (
    State_id INT,
    Year INT CHECK (Year BETWEEN 2016 AND 2024),
    Total_businesses INT,	
    Business_entries INT,	
    Business_exits	INT,
    Elec_relevant_businesses INT,
    NonResidential_building_value_m	INT,
    Elec_relevant_debtors INT,
    Elec_relevant_employee_per FLOAT,
    primary key (State_id, Year),
    FOREIGN KEY (State_id) REFERENCES States(State_id)
);

CREATE TABLE Fact_projects (
    Project_id INT primary key,
    Power_station_name VARCHAR(100), 
    State_id INT,
    Installed_capacity FLOAT,
    Postcode INT CHECK (Postcode BETWEEN 1000 AND 9999),
    Fuel_id INT,
    geom GEOMETRY,
    FOREIGN KEY (Fuel_id) REFERENCES Primary_fuel(Primary_fuel_id),
    FOREIGN KEY (State_id) REFERENCES States(State_id)
);

CREATE TABLE Fact_Emission (
    Emission_id INT primary key,
    State_id INT,
    Facility_id INT, 
    Electricity_production_MWh NUMERIC,
    Total_emission_tCO2e NUMERIC,
    Emission_intensity_tCO2e_MWh FLOAT,
    Grid_id INT,
    Primary_fuel_id INT,
    Year INT CHECK (Year BETWEEN 2014 AND 2024),
    FOREIGN KEY (Facility_id) REFERENCES Facility(Facility_id),
    FOREIGN KEY (Grid_id) REFERENCES Grid(Grid_id),
    FOREIGN KEY (Primary_fuel_id) REFERENCES Primary_fuel(Primary_fuel_id),
    FOREIGN KEY (State_id, Year) REFERENCES Economic_status(State_id, Year)
);

In [ ]:
%%sql
--insert data in each relation from cleaned data
INSERT INTO States
SELECT 
    row_number() OVER () - 1,
    c1,
    c2
FROM (VALUES
        ('SA', 'South Australia'),
        ('NSW', 'New South Wales'),
        ('VIC', 'Victoria'),
        ('QLD', 'Queensland'),
        ('WA', 'Western Australia'),
        ('ACT', 'Australian Capital Territory'),
        ('NT', 'Northern Territory'),
        ('TAS', 'Tasmania'),
        ('OTHER', 'Other Territories')
) AS t(c1, c2);
SELECT * FROM States;

In [ ]:
%%sql
INSERT INTO Facility
SELECT
    row_number() OVER () - 1,
    facilityname,
    reportingentity
FROM nger;
select * from Facility limit 5;

In [ ]:
%%sql
INSERT INTO Primary_fuel
SELECT
    row_number() OVER () - 1,
    fuel,
FROM (
    SELECT primaryfuel AS fuel FROM nger
    UNION
    SELECT "fuel source (s)" AS fuel FROM cer);
select * from Primary_fuel limit 5;


In [ ]:
%%sql
select * from Primary_fuel where Primary_fuel_id >= 10;

In [ ]:
%%sql
INSERT INTO Grid
SELECT row_number() OVER () - 1, * FROM (VALUES ('NEM'),('SWIS'),('Mt Isa'),('Off-grid'),('NWIS'),('DKIS'));
select * from Grid;

In [ ]:
%%sql
INSERT INTO Economic_status
SELECT
    States.State_id,
    abs.Year,
    abs."Total number of businesses",
    abs."Total number of business entries",
    abs."Total number of business exits",
    abs."Electricity, gas, water and waste services (no.)",
    abs."Value of non-residential building ($m)",
    abs."Technicians and trades workers (no.)",
    abs."Electricity, gas, water and waste services (%)"
FROM abs, States
WHERE abs.Label = States.State_full;
SELECT * FROM Economic_status where Total_businesses IS NOT NULL LIMIT 5;

In [ ]:
%%sql
INSERT INTO Fact_projects
SELECT
    ROW_NUMBER() OVER (ORDER BY cer.power_station_name_clean) - 1 AS Project_id,
    cer.power_station_name_clean,
    States.State_id,
    cer."installed capacity (mw)",
    cer.postcode,
    Primary_fuel.Primary_fuel_id,
    cer.geom
FROM cer JOIN States ON cer.state_full = States.State_full
JOIN Primary_fuel ON cer."fuel source (s)" = Primary_fuel.Primary_fuel_name;
SELECT * FROM Fact_projects LIMIT 5;

In [ ]:
%%sql
INSERT INTO Fact_Emission
SELECT
    row_number() OVER () - 1,
    States.State_id,
    Facility.Facility_id,
    nger.electricityproductionmwh,
    nger.totalemissionstCO2e,
    nger.emissionintensitytmwh,
    Grid.Grid_id,
    Primary_fuel.Primary_fuel_id,
    nger.year
FROM nger
JOIN States ON nger.State = States.State_abbr
JOIN Facility ON nger.facilityname = Facility.Facility_name
JOIN Grid ON nger.grid = Grid.Grid_name
JOIN Primary_fuel ON nger.primaryfuel = Primary_fuel.Primary_fuel_name;
SELECT * FROM Fact_Emission LIMIT 5;